# Módulo 2.2 + 2.3 — De ecuaciones a escenarios
## Explotación de la Información · Hotel Booking Demand

En esta clase no vamos a separar artificialmente "ecuaciones" y "sistemas estocásticos".

Vamos a seguir una sola pregunta:

> **¿Cómo puedo transformar los datos de reservaciones en escenarios útiles para tomar decisiones?**

La ruta será:

**DATOS → ECUACIONES → INDICADORES → ESCENARIOS → INCERTIDUMBRE**

---

### 2.2 aporta
Reglas de cálculo que transforman columnas en información nueva.

### 2.3 aporta
La idea de que algunos resultados no son deterministas: una reservación puede cancelar o no.

El objetivo no es hacer estadística formal.  
El objetivo es **explotar el dataset para construir escenarios interpretables**.


# 0. Cargar el dataset


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

paths = [
    Path("hotel_bookings_clean.csv"),
    Path("/content/hotel_bookings_clean.csv"),
    Path("/mnt/data/hotel_bookings_clean.csv")
]

DATA_PATH = next((p for p in paths if p.exists()), None)

if DATA_PATH is None:
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = Path(next(iter(uploaded.keys())))

df = pd.read_csv(DATA_PATH)

print("Filas:", len(df))
print("Columnas:", len(df.columns))
display(df.head())


# 1. Construir información que no existe directamente en el dataset

Una fila contiene piezas de información.

Nosotros podemos combinarlas mediante ecuaciones.

## Duración total

$
total\_nights =
stays\_in\_week\_nights +
stays\_in\_weekend\_nights
$

## Tamaño del grupo

$
total\_guests =
adults + children + babies
$

## Valor tarifario estimado de la reservación

$
booking\_value =
ADR \times total\_nights
$

> `booking_value` no representa ingreso contable real.  
> Es una medida derivada para comparar reservaciones.


In [ ]:
df["total_nights"] = (
    df["stays_in_week_nights"] +
    df["stays_in_weekend_nights"]
)

df["total_guests"] = (
    df["adults"] +
    df["children"] +
    df["babies"]
)

df["booking_value"] = (
    df["adr"] *
    df["total_nights"]
)

df[
    ["total_nights", "total_guests", "booking_value"]
].describe()


# 2. Separar valor asociado a reservaciones canceladas y no canceladas

Podemos construir:

$
cancelled\_booking\_value =
booking\_value \times is\_canceled
$

$
retained\_booking\_value =
booking\_value \times (1-is\_canceled)
$

Esto permite responder una pregunta nueva:

> **¿Qué parte del valor tarifario estimado está asociada con reservas que terminaron cancelándose?**


In [ ]:
df["cancelled_booking_value"] = (
    df["booking_value"] *
    df["is_canceled"]
)

df["retained_booking_value"] = (
    df["booking_value"] *
    (1 - df["is_canceled"])
)

summary = pd.Series({
    "Valor tarifario total estimado":
        df["booking_value"].sum(),

    "Valor asociado a cancelaciones":
        df["cancelled_booking_value"].sum(),

    "Valor asociado a no canceladas":
        df["retained_booking_value"].sum(),

    "Reservaciones":
        int(len(df)),

    "Cancelaciones":
        int(df["is_canceled"].sum())
})

summary


## Pregunta para clase

¿Es mejor decir:

> "27% de las reservas se cancelaron"

o también poder decir:

> "las reservas canceladas representan X unidades del valor tarifario estimado del dataset"?

La segunda medida incorpora **magnitud**, no solo conteo.


# 3. Crear indicadores útiles a partir de ecuaciones

Ahora podemos construir varias medidas que no estaban originalmente en la tabla.


In [ ]:
indicators = pd.Series({
    "ADR promedio":
        df["adr"].mean(),

    "Noches promedio":
        df["total_nights"].mean(),

    "Huéspedes promedio":
        df["total_guests"].mean(),

    "Valor promedio por reserva":
        df["booking_value"].mean(),

    "Valor promedio por huésped":
        (
            df.loc[df["total_guests"] > 0, "booking_value"] /
            df.loc[df["total_guests"] > 0, "total_guests"]
        ).mean(),

    "Cancelación (%)":
        100 * df["is_canceled"].mean(),

    "Valor asociado a cancelaciones (%)":
        100 * df["cancelled_booking_value"].sum() /
        df["booking_value"].sum()
})

indicators


# 4. Escenario determinista 1 — ¿Qué pasaría si ADR aumentara 5%?

Una ecuación permite crear escenarios.

$
ADR_{nuevo}=ADR\times1.05
$

$
booking\_value_{nuevo}=ADR_{nuevo}\times total\_nights
$

Aquí no hay azar.

Si mantenemos las mismas reservaciones y aumentamos ADR 5%, el resultado está completamente determinado por la ecuación.


In [ ]:
df["adr_plus_5"] = df["adr"] * 1.05

df["booking_value_plus_5"] = (
    df["adr_plus_5"] *
    df["total_nights"]
)

base_value = df["booking_value"].sum()
new_value = df["booking_value_plus_5"].sum()

print("Valor base:", round(base_value, 2))
print("Valor con ADR +5%:", round(new_value, 2))
print("Diferencia:", round(new_value - base_value, 2))


# 5. Escenario determinista 2 — Recuperar parte de las cancelaciones

Supongamos un escenario hipotético:

> Una estrategia logra retener 10% de las reservas que históricamente terminaron cancelándose.

Entonces:

$
valor\_recuperado =
0.10 \times cancelled\_booking\_value
$

No estamos afirmando que la estrategia exista.

Estamos usando el dataset para cuantificar:

> **¿qué magnitud representaría ese escenario?**


In [ ]:
cancelled_value = df["cancelled_booking_value"].sum()

scenario_rates = [0.05, 0.10, 0.20]

scenario_table = pd.DataFrame({
    "Porcentaje recuperado": scenario_rates
})

scenario_table["Valor estimado recuperado"] = (
    scenario_table["Porcentaje recuperado"] *
    cancelled_value
)

scenario_table["Porcentaje recuperado"] *= 100

scenario_table


In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    scenario_table["Porcentaje recuperado"].astype(str) + "%",
    scenario_table["Valor estimado recuperado"]
)

plt.xlabel("Reservas canceladas recuperadas")
plt.ylabel("Valor tarifario estimado")
plt.title("Escenarios de recuperación")
plt.grid(axis="y", alpha=0.25)

plt.show()


# 6. Lo determinista tiene un límite

Hasta ahora todas las ecuaciones producen un resultado exacto porque nosotros fijamos las condiciones.

Pero hay una parte del proceso que no conocemos de antemano:

> **¿qué reservaciones futuras cancelarán?**

Sabemos lo que ocurrió históricamente.

No sabemos exactamente qué ocurrirá en la siguiente reserva.

Aquí empieza la parte **estocástica**.


# 7. Convertir comportamiento histórico en una referencia

La tasa histórica global es:

$
P(Cancelar) =
\frac{cancelaciones}{reservaciones}
$

La usaremos como una referencia de comportamiento.

No es una predicción perfecta.


In [ ]:
p_cancel = df["is_canceled"].mean()

print(
    "Tasa histórica de cancelación:",
    f"{100*p_cancel:.2f}%"
)


# 8. Primera simulación — 100 reservaciones

Imaginemos 100 nuevas reservaciones con una probabilidad de cancelación similar a la histórica.

Cada reservación puede tomar uno de dos estados:

- `0` → no cancela
- `1` → cancela

Ejecuta varias veces la siguiente celda.


In [ ]:
rng = np.random.default_rng()

simulated_cancel = rng.choice(
    [0, 1],
    size=100,
    p=[1-p_cancel, p_cancel]
)

print(
    "Cancelaciones simuladas:",
    simulated_cancel.sum()
)


### Pregunta

Si la tasa histórica es cercana a 27%:

> ¿por qué no obtenemos exactamente 27 cancelaciones cada vez?

Porque una tasa describe un comportamiento general.

El resultado concreto puede variar.

Eso es justamente lo que queremos representar con un **sistema estocástico**.


# 9. Repetir el escenario muchas veces

En vez de ejecutar manualmente una simulación, podemos repetirla muchas veces.

Pregunta:

> **Si recibimos bloques de 100 reservaciones, qué resultados podríamos observar?**


In [ ]:
rng = np.random.default_rng(42)

simulations = []

for _ in range(1000):
    result = rng.choice(
        [0, 1],
        size=100,
        p=[1-p_cancel, p_cancel]
    )

    simulations.append(result.sum())

simulations = np.array(simulations)

print(
    "Cancelaciones promedio:",
    round(simulations.mean(), 1)
)

print(
    "Mínimo observado:",
    simulations.min()
)

print(
    "Máximo observado:",
    simulations.max()
)


In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    simulations,
    bins=20
)

plt.axvline(
    simulations.mean(),
    linestyle="--",
    label=f"Promedio ≈ {simulations.mean():.1f}"
)

plt.xlabel("Cancelaciones en 100 reservas")
plt.ylabel("Número de simulaciones")
plt.title("Resultados posibles para 100 reservaciones")
plt.legend()
plt.grid(alpha=0.2)

plt.show()


# 10. El contexto importa

Usar una sola tasa global simplifica demasiado.

El dataset muestra que la cancelación cambia según el segmento de mercado.

Podemos usar esa información para construir un escenario más realista.


In [ ]:
segment_rates = (
    df.groupby("market_segment")
      ["is_canceled"]
      .mean()
      .sort_values(ascending=False)
)

(segment_rates * 100).round(1)


# 11. Comparar dos escenarios de 100 reservaciones

Vamos a comparar:

- 100 reservas **Online TA**
- 100 reservas **Direct**

usando sus tasas históricas.


In [ ]:
p_online = segment_rates["Online TA"]
p_direct = segment_rates["Direct"]

rng = np.random.default_rng(123)

online_results = rng.binomial(
    n=100,
    p=p_online,
    size=1000
)

direct_results = rng.binomial(
    n=100,
    p=p_direct,
    size=1000
)

print(
    "Online TA:",
    f"{online_results.mean():.1f} cancelaciones promedio / 100"
)

print(
    "Direct:",
    f"{direct_results.mean():.1f} cancelaciones promedio / 100"
)


In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    online_results,
    bins=20,
    alpha=0.55,
    label="Online TA"
)

plt.hist(
    direct_results,
    bins=20,
    alpha=0.55,
    label="Direct"
)

plt.xlabel("Cancelaciones en 100 reservas")
plt.ylabel("Simulaciones")
plt.title("Dos escenarios con comportamientos históricos distintos")
plt.legend()
plt.grid(alpha=0.2)

plt.show()


# 12. Juntar ecuaciones + incertidumbre

Hasta ahora simulamos únicamente cuántas reservas cancelan.

Ahora podemos combinar:

$
booking\_value = ADR\times nights
$

con:

$
cancelación = evento\ aleatorio
$

y preguntar:

> **¿cuánto valor tarifario podría conservarse en diferentes escenarios de cancelación?**


In [ ]:
# Tomamos una muestra de 500 reservaciones como cartera hipotética.
portfolio = (
    df.sample(
        500,
        random_state=42
    )
    .copy()
)

portfolio_value = portfolio["booking_value"].sum()

print(
    "Valor tarifario estimado de la cartera:",
    round(portfolio_value, 2)
)


# 13. Simular el valor conservado de una cartera

Para simplificar, utilizaremos la tasa histórica global.

En cada simulación:

1. cada reserva puede cancelar o no;
2. si cancela, no contamos su `booking_value`;
3. sumamos el valor de las no canceladas.

Ejecutamos el mismo escenario 500 veces.


In [ ]:
rng = np.random.default_rng(2026)

retained_values = []

for _ in range(500):

    simulated_cancel = rng.binomial(
        n=1,
        p=p_cancel,
        size=len(portfolio)
    )

    retained = (
        portfolio["booking_value"].to_numpy() *
        (1 - simulated_cancel)
    ).sum()

    retained_values.append(retained)

retained_values = np.array(retained_values)

print(
    "Valor promedio conservado:",
    round(retained_values.mean(), 2)
)

print(
    "Menor escenario simulado:",
    round(retained_values.min(), 2)
)

print(
    "Mayor escenario simulado:",
    round(retained_values.max(), 2)
)


In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    retained_values,
    bins=25
)

plt.axvline(
    retained_values.mean(),
    linestyle="--",
    label="Promedio simulado"
)

plt.xlabel("Valor tarifario conservado")
plt.ylabel("Simulaciones")
plt.title("500 posibles resultados de la misma cartera")
plt.legend()
plt.grid(alpha=0.2)

plt.show()


# 14. ¿Qué aprendimos?

La cartera de reservaciones era exactamente la misma.

Sin embargo, el valor final cambió entre simulaciones porque:

> **no sabemos de antemano cuáles reservas cancelarán.**

Aquí se unen los dos temas:

### Ecuación estructurada

$
booking\_value = ADR\times nights
$

### Componente estocástico

$
cancelación \in \{0,1\}
$

### Resultado

$
retained\_value =
booking\_value\times(1-cancelación)
$

Una ecuación puede contener variables cuyo resultado no conocemos de antemano.


# 15. Actividad — Construye un escenario con tu propio proyecto

No quiero que repitas exactamente el caso del hotel.

Busca en tu dataset:

## A. Una cantidad que puedas construir con una ecuación

Ejemplos:

- costo total;
- duración total;
- consumo;
- rendimiento;
- valor;
- tasa;
- diferencia;
- utilización.

Debes explicar:

> ¿qué información nueva produce esta ecuación?

## B. Un evento o condición incierta

Ejemplos:

- compra / no compra;
- falla / no falla;
- abandono / permanencia;
- retraso / puntualidad;
- fraude / normal;
- aprobación / rechazo.

## C. Un escenario

Combina ambos elementos.

Pregunta:

> **¿cómo cambia el resultado calculado cuando el evento incierto ocurre con diferentes frecuencias?**

---

### Entrega

1. fórmula;
2. significado;
3. escenario base;
4. escenario alternativo;
5. una visualización;
6. tres conclusiones.


# Cierre

No necesitamos tratar 2.2 y 2.3 como temas aislados.

Podemos verlos como dos capas de un mismo análisis:

## 2.2 — construir

$
DATOS
\rightarrow
ECUACIONES
\rightarrow
INFORMACIÓN
$

## 2.3 — explorar lo que puede variar

$
INFORMACIÓN
+
INCERTIDUMBRE
\rightarrow
ESCENARIOS
$

## Resultado

$
\boxed{
DATOS
\rightarrow
ECUACIONES
\rightarrow
ESCENARIOS
\rightarrow
DECISIÓN
}
$

> Explotar datos es construir información útil y explorar qué podría cambiar cuando las condiciones no son completamente conocidas.
